In [ ]:
import hmac, hashlib, base64, json, requests
from datetime import datetime

# Använd dina värden här
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e" # BYT UT DENNA i portalen sen!
BASE_URL = "https://www.soliscloud.com:13333"

def test_connection():
    path = "/v1/api/userStationList" # En enklare endpoint för test
    body = json.dumps({"pageNo": 1, "pageSize": 10})
    
    content_md5 = base64.b64encode(hashlib.md5(body.encode('utf-8')).digest()).decode('utf-8')
    content_type = "application/json;charset=UTF-8"
    date_str = datetime.utcnow().strftime("%a, %d %b %Y %H:%M:%S GMT")
    sign_str = f"POST\n{content_md5}\n{content_type}\n{date_str}\n{path}"
    
    hash_obj = hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1)
    sign = base64.b64encode(hash_obj.digest()).decode('utf-8')
    
    headers = {
        "Content-Type": content_type,
        "Content-MD5": content_md5,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }
    
    resp = requests.post(BASE_URL + path, headers=headers, data=body)
    print(f"Statuskod: {resp.status_code}")
    print(f"Svar från Solis: {resp.text}")

test_connection()

In [7]:
import hmac
import hashlib
import base64
import json
import requests
import pandas as pd
from datetime import datetime, timezone

# --- 1. KONFIGURATION ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e" # Kom ihåg att byta denna i portalen sen!
INVERTER_SN = "1805040227120162"

# Vi backar till www-adressen som vi vet får kontakt
BASE_URL = "https://www.soliscloud.com:13333" 

def get_solis_headers(path, body):
    """Skapar headers med en signatur som tvingar engelska datum."""
    content_md5 = base64.b64encode(hashlib.md5(body.encode('utf-8')).digest()).decode('utf-8')
    content_type = "application/json;charset=UTF-8"
    
    # Fix för DeprecationWarning och språk (tvingar engelska namn)
    now = datetime.now(timezone.utc)
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    date_str = f"{days[now.weekday()]}, {now.day:02d} {months[now.month-1]} {now.year} {now.strftime('%H:%M:%S')} GMT"
    
    # Signatursträngen
    sign_str = f"POST\n{content_md5}\n{content_type}\n{date_str}\n{path}"
    
    hash_obj = hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1)
    sign = base64.b64encode(hash_obj.digest()).decode('utf-8')
    
    return {
        "Content-Type": content_type,
        "Content-MD5": content_md5,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }

def fetch_data(date_str):
    path = "/v1/api/inverterDay"
    payload = {
        "sn": INVERTER_SN,
        "time": date_str,
        "timeZone": 2 
    }
    body = json.dumps(payload)
    headers = get_solis_headers(path, body)
    
    try:
        response = requests.post(BASE_URL + path, headers=headers, data=body, timeout=15)
        return response.json()
    except Exception as e:
        return {"code": "-1", "msg": str(e)}

# --- 2. EXEKVERING ---
all_days_data = []

for day in range(1, 8):
    target_date = f"2025-07-{day:02d}"
    print(f"Hämtar {target_date}...", end=" ")
    
    result = fetch_data(target_date)
    
    # Vi kollar både siffran 0 och strängen "0" för säkerhets skull
    if str(result.get("code")) == "0" and result.get("data"):
        all_days_data.extend(result["data"])
        print("✅ Success!")
    else:
        print(f"❌ Fel: {result.get('msg', 'Ingen data')}")

# --- 3. SPARA TILL CSV ---
if all_days_data:
    df = pd.DataFrame(all_days_data)
    
    # Konvertera tid (Solis skickar oftast 'dataTimestamp')
    if 'dataTimestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['dataTimestamp'], unit='ms')
    elif 'time' in df.columns:
        df['timestamp'] = pd.to_datetime(df['time'], unit='ms')
        
    # Försök hitta effekt-kolumnen (kan heta 'power' eller 'activePower')
    power_col = 'activePower' if 'activePower' in df.columns else 'power'
    
    if 'timestamp' in df.columns and power_col in df.columns:
        df[power_col] = pd.to_numeric(df[power_col], errors='coerce')
        df.set_index('timestamp', inplace=True)
        
        # Sammanställ till timme
        df_hourly = df[power_col].resample('1H').mean().reset_index()
        df_hourly.to_csv("soliscloud_hourly_data_1.csv", index=False)
        print(f"\nKlart! Filen är skapad med {len(df_hourly)} rader.")
        display(df_hourly.head())
    else:
        print("\nKunde inte hitta rätt kolumner i datan. Kolumner som finns:", df.columns.tolist())
else:
    print("\nIngen data samlades in.")

Hämtar 2025-07-01... ❌ Fel: Ingen data
Hämtar 2025-07-02... ❌ Fel: Ingen data
Hämtar 2025-07-03... ❌ Fel: Ingen data
Hämtar 2025-07-04... ❌ Fel: Ingen data
Hämtar 2025-07-05... ❌ Fel: Ingen data
Hämtar 2025-07-06... ❌ Fel: Ingen data
Hämtar 2025-07-07... ❌ Fel: Ingen data

Ingen data samlades in.


In [9]:
import hmac
import hashlib
import base64
import json
import requests
from datetime import datetime, timezone

# --- DINA UPPGIFTER ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e" 
BASE_URL = "https://www.soliscloud.com:13333"

def get_solis_auth(path, payload_dict):
    # 1. Skapa JSON utan några mellanslag (viktigt för signaturen!)
    body = json.dumps(payload_dict, separators=(',', ':'))
    
    # 2. MD5 hash
    content_md5 = base64.b64encode(hashlib.md5(body.encode('utf-8')).digest()).decode('utf-8')
    content_type = "application/json;charset=UTF-8"
    
    # 3. Datum (måste vara exakt RFC 1123 format)
    now = datetime.now(timezone.utc)
    date_str = now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    # 4. Skapa signatursträngen
    sign_str = f"POST\n{content_md5}\n{content_type}\n{date_str}\n{path}"
    
    # 5. HMAC-SHA1
    hash_obj = hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1)
    sign = base64.b64encode(hash_obj.digest()).decode('utf-8')
    
    headers = {
        "Content-MD5": content_md5,
        "Content-Type": content_type,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }
    return headers, body

# --- TESTANROP ---
path = "/v1/api/userStationList"
payload = {"pageNo": 1, "pageSize": 10}

headers, clean_body = get_solis_auth(path, payload)
response = requests.post(BASE_URL + path, headers=headers, data=clean_body)

print(f"Status: {response.status_code}")
print("Svar:", json.dumps(response.json(), indent=2))

Status: 403
Svar: {
  "timestamp": 1774527952503,
  "status": 403,
  "error": "Forbidden",
  "message": "wrong sign",
  "path": "/v1/api/userStationList"
}


In [11]:
import hmac, hashlib, base64, json, requests
from datetime import datetime, timezone

# --- DINA UPPGIFTER ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e" 
BASE_URL = "https://www.soliscloud.com:13333"

path = "/v1/api/userStationList"
payload = {"pageNo": 1, "pageSize": 10}
body = json.dumps(payload, separators=(',', ':'))

# Skapa signatur komponenter
content_md5 = base64.b64encode(hashlib.md5(body.encode('utf-8')).digest()).decode('utf-8')
date_str = datetime.now(timezone.utc).strftime('%a, %d %b %Y %H:%M:%S GMT')
sign_str = f"POST\n{content_md5}\napplication/json;charset=UTF-8\n{date_str}\n{path}"

# Signera
sign = base64.b64encode(hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1).digest()).decode('utf-8')

headers = {
    "Content-MD5": content_md5,
    "Content-Type": "application/json;charset=UTF-8",
    "Date": date_str,
    "Authorization": f"API {API_ID}:{sign}"
}

print("--- DEBUG INFO ---")
print(f"URL: {BASE_URL}{path}")
print(f"Date Header: {date_str}")
print(f"Content-MD5: {content_md5}")
print(f"Authorization: API {API_ID}:[SIGNATUR]")
print("------------------")

res = requests.post(BASE_URL + path, headers=headers, data=body)
print(f"SVAR FRÅN SERVER ({res.status_code}):")
print(res.text)

--- DEBUG INFO ---
URL: https://www.soliscloud.com:13333/v1/api/userStationList
Date Header: Thu, 26 Mar 2026 12:58:47 GMT
Content-MD5: kxdxk7rbAsrzSIWgEwhH4w==
Authorization: API 1300386381676697014:[SIGNATUR]
------------------
SVAR FRÅN SERVER (403):
{
  "timestamp" : 1774529927622,
  "status" : 403,
  "error" : "Forbidden",
  "message" : "wrong sign",
  "path" : "/v1/api/userStationList"
}


In [9]:
import hmac
import hashlib
import base64
import json
import requests
from datetime import datetime, timezone

# --- KONFIGURATION ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e".strip() # .strip() tar bort ev. osynliga tecken
INVERTER_SN = "1805040227120162"
BASE_URL = "https://www.soliscloud.com:13333"

def fetch_solis_day(target_date):
    path = "v1/app/inverterDay"
    
    # Payload MÅSTE vara exakt så här, utan mellanslag i JSON-strängen
    payload = {
        "sn": INVERTER_SN,
        "time": target_date,
        "timeZone": 2
    }
    body = json.dumps(payload, separators=(',', ':'))
    
    # 1. MD5
    content_md5 = base64.b64encode(hashlib.md5(body.encode('utf-8')).digest()).decode('utf-8')
    
    # 2. Datum (Solis kräver engelska och exakt format)
    now = datetime.now(timezone.utc)
    date_str = now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    # 3. Signatur (Metod + MD5 + ContentType + Date + Path)
    content_type = "application/json;charset=UTF-8"
    sign_str = f"POST\n{content_md5}\n{content_type}\n{date_str}\n{path}"
    
    # 4. HMAC-SHA1
    sign = base64.b64encode(hmac.new(
        API_SECRET.encode('utf-8'), 
        sign_str.encode('utf-8'), 
        hashlib.sha1
    ).digest()).decode('utf-8')
    
    headers = {
        "Content-MD5": content_md5,
        "Content-Type": content_type,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }
    
    print(f"Anropar {path} för datum {target_date}...")
    response = requests.post(BASE_URL + path, headers=headers, data=body)
    return response

# --- KÖR TESTET ---
res = fetch_solis_day("2025-07-01")

print(f"Statuskod: {res.status_code}")
try:
    data = res.json()
    print("Svar från Solis:", json.dumps(data, indent=2))
except:
    print("Kunde inte tolka svaret som JSON:", res.text)

Anropar v1/app/inverterDay för datum 2025-07-01...


InvalidURL: Failed to parse: https://www.soliscloud.com:13333v1/app/inverterDay

In [7]:
import requests
from datetime import datetime, timezone

def check_time_drift():
    # Vi hämtar tiden från Google (som alltid är rätt)
    res = requests.get("https://www.google.com")
    server_date_str = res.headers['Date'] # Format: 'Thu, 26 Mar 2026 12:45:00 GMT'
    
    local_now = datetime.now(timezone.utc)
    local_date_str = local_now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    print(f"Server-tid (Google): {server_date_str}")
    print(f"Din lokala tid:      {local_date_str}")

check_time_drift()

Server-tid (Google): Thu, 26 Mar 2026 12:48:32 GMT
Din lokala tid:      Thu, 26 Mar 2026 12:48:31 GMT


In [12]:
import hmac, hashlib, base64, json, requests
from datetime import datetime, timezone

# --- KONFIGURATION ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e"
INVERTER_SN = "1805040227120162"
BASE_URL = "https://www.soliscloud.com:13333"

def fetch_with_user_specs(path, payload):
    # 1. Body och MD5
    body_str = json.dumps(payload, separators=(',', ':'))
    md5_hash = hashlib.md5(body_str.encode('utf-8')).digest()
    content_md5 = base64.b64encode(md5_hash).decode('utf-8')
    
    # 2. Datum (Standard GMT)
    now = datetime.now(timezone.utc)
    gmt_date = now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    # 3. Signatur-strängen (Enligt din spec: application/json utan charset)
    sign_content_type = "application/json"
    
    # Vi bygger strängen med riktiga newlines
    sign_str = f"POST\n{content_md5}\n{sign_content_type}\n{gmt_date}\n{path}"
    
    # Debug: Skriv ut signatursträngen för att kontrollera radbrytningar
    # print("--- DEBUG: SIGN_STR ---")
    # print(repr(sign_str)) 
    
    # 4. Signering
    hashed = hmac.new(API_SECRET.encode('utf-8'), 
                      sign_str.encode('utf-8'), 
                      hashlib.sha1).digest()
    sign = base64.b64encode(hashed).decode('utf-8')
    
    # 5. Headers
    headers = {
        "Content-MD5": content_md5,
        "Content-Type": "application/json;charset=UTF-8", # Full i headern
        "Date": gmt_date,
        "Authorization": f"API {API_ID}:{sign}" # Ingen blank efter kolonet här
    }
    
    response = requests.post(BASE_URL + path, headers=headers, data=body_str, timeout=10)
    return response, sign_str, content_md5

# --- EXEKVERING ---
target_path = "/v1/api/inverterDay"

# Uppdaterad payload med 'money' och negativ timeZone för Sverige (UTC+2 i juli -> -2)
target_payload = {
    "sn": INVERTER_SN,
    "money": "EUR",      
    "time": "2025-07-01",
    "timeZone": -2       
}

res, s_str, c_md5 = fetch_with_user_specs(target_path, target_payload)

print(f"Status: {res.status_code}")
print(f"Svar: {res.text}")

if res.status_code != 200 or "wrong sign" in res.text:
    print("\n--- FELSÖKNINGSDATA ---")
    print(f"Content-MD5: {c_md5}")
    print(f"Date: {res.request.headers['Date']}")
    print(f"Payload: {json.dumps(target_payload, separators=(',', ':'))}")

Status: 200
Svar: {
  "success" : true,
  "code" : "0",
  "msg" : "success",
  "data" : [ {
    "dataTimestamp" : "1751336400848",
    "timeStr" : "2025-07-01 00:20:00",
    "acOutputType" : 1,
    "dcInputType" : 1,
    "state" : 1,
    "time" : "10:20:00",
    "pac" : 20.000,
    "pacStr" : "kW",
    "pacPec" : "0.001",
    "eToday" : 0,
    "eTotal" : 23097.000,
    "uPv1" : 320.0,
    "iPv1" : 0.2,
    "uPv2" : 351.2,
    "iPv2" : 0.1,
    "uPv3" : 0,
    "iPv3" : 0,
    "uPv4" : 0,
    "iPv4" : 0,
    "uPv5" : 0,
    "iPv5" : 0,
    "uPv6" : 0,
    "iPv6" : 0,
    "uPv7" : 0,
    "iPv7" : 0,
    "uPv8" : 0,
    "iPv8" : 0,
    "uPv9" : 0,
    "iPv9" : 0,
    "uPv10" : 0,
    "iPv10" : 0,
    "uPv11" : 0,
    "iPv11" : 0,
    "uPv12" : 0,
    "iPv12" : 0,
    "uPv13" : 0,
    "iPv13" : 0,
    "uPv14" : 0,
    "iPv14" : 0,
    "uPv15" : 0,
    "iPv15" : 0,
    "uPv16" : 0,
    "iPv16" : 0,
    "uPv17" : 0,
    "iPv17" : 0,
    "uPv18" : 0,
    "iPv18" : 0,
    "uPv19" : 0,
    "iPv1

In [13]:
import hmac, hashlib, base64, json, requests, time
import pandas as pd
from datetime import datetime, timezone, timedelta

# --- 1. KONFIGURATION ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e"
INVERTER_SN = "1805040227120162"
BASE_URL = "https://www.soliscloud.com:13333"

def get_solis_headers(path, body_dict):
    body_str = json.dumps(body_dict, separators=(',', ':'))
    md5_hash = hashlib.md5(body_str.encode('utf-8')).digest()
    content_md5 = base64.b64encode(md5_hash).decode('utf-8')
    
    # Datum i GMT (RFC 1123)
    now = datetime.now(timezone.utc)
    date_str = now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    content_type = "application/json;charset=UTF-8"
    sign_content_type = "application/json"
    
    # --- KORRIGERAD SIGN_STR: Vi använder join för att vara 100% säkra på radbrytningar ---
    sign_parts = [
        "POST",
        content_md5,
        sign_content_type,
        date_str,
        path
    ]
    sign_str = "\n".join(sign_parts)
    
    hashed = hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1).digest()
    sign = base64.b64encode(hashed).decode('utf-8')
    
    headers = {
        "Content-MD5": content_md5,
        "Content-Type": content_type,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }
    return headers, body_str

def fetch_day(target_date, tz_offset):
    path = "/v1/api/inverterDay"
    payload = {
        "sn": INVERTER_SN,
        "money": "EUR",
        "time": target_date,
        "timeZone": tz_offset  # Använder positiv offset enligt dokumentation
    }
    headers, body = get_solis_headers(path, payload)
    try:
        res = requests.post(BASE_URL + path, headers=headers, data=body, timeout=15)
        if res.status_code == 200:
            return res.json()
    except Exception as e:
        print(f"\nFel vid anrop för {target_date}: {e}")
    return None

# --- 2. LOOP FÖR JULI - DECEMBER 2023 ---
start_date = datetime(2023, 7, 1)
end_date = datetime(2023, 12, 31)
dst_end_2023 = datetime(2023, 10, 29) # Dagen då vi går från UTC+2 till UTC+1

current_date = start_date
all_frames = []

print(f"Hämtar data för perioden {start_date.date()} till {end_date.date()}...")

while current_date <= end_date:
    date_str = current_date.strftime('%Y-%m-%d')
    
    # Bestäm tidszon för datumet
    tz = 2 if current_date < dst_end_2023 else 1
    
    print(f"Hämtar: {date_str} (TZ: {tz})", end="\r")
    
    result = fetch_day(date_str, tz)
    
    if result and str(result.get("code")) == "0":
        day_data = result.get("data", [])
        if day_data:
            all_frames.append(pd.DataFrame(day_data))
    
    current_date += timedelta(days=1)
    time.sleep(0.4) # Snabbare men säkert

# --- 3. SAMMANSTÄLLNING ---
if all_frames:
    full_df = pd.concat(all_frames, ignore_index=True)
    
    # Konvertera tid. 'dataTimestamp' är säkrast.
    full_df['timestamp'] = pd.to_datetime(full_df['dataTimestamp'].astype(float), unit='ms')
    full_df['pac'] = pd.to_numeric(full_df['pac'], errors='coerce')
    full_df.set_index('timestamp', inplace=True)
    
    # Aggregera till timvärden (medel-kW under timmen)
    hourly_df = full_df['pac'].resample('1H').mean().reset_index()
    hourly_df.columns = ['timestamp', 'avg_power_kw']
    
    # Spara
    output_file = "solis_data_2023.csv"
    hourly_df.to_csv(output_file, index=False)
    
    print(f"\n\nKlart! {len(all_frames)} dagar med data hittades.")
    print(f"Resultatet har sparats i {output_file}")
    display(hourly_df.head(10))
else:
    print("\nIngen data hittades. Kontrollera serienummer och API-status.")

Hämtar data för perioden 2023-07-01 till 2023-12-31...
Hämtar: 2023-12-31 (TZ: 1)

Klart! 184 dagar med data hittades.
Resultatet har sparats i solis_data_2023.csv


C:\Users\micha\AppData\Local\Temp\ipykernel_52932\3406910485.py:99: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_df = full_df['pac'].resample('1H').mean().reset_index()


,timestamp,avg_power_kw
0,2023-07-01 02:00:00,23.333333
1,2023-07-01 03:00:00,176.666667
2,2023-07-01 04:00:00,239.166667
3,2023-07-01 05:00:00,253.333333
4,2023-07-01 06:00:00,752.500000
5,2023-07-01 07:00:00,655.833333
6,2023-07-01 08:00:00,975.833333
7,2023-07-01 09:00:00,824.166667
8,2023-07-01 10:00:00,566.666667
9,2023-07-01 11:00:00,609.166667


In [14]:
import pandas as pd

# Läs in den gamla filen
df_2023 = pd.read_csv('solis_data_2023.csv')

# Konvertera timestamp till svensk tid
# Vi antar att de befintliga tiderna i CSV-filen sparades som UTC
df_2023['timestamp'] = pd.to_datetime(df_2023['timestamp'], utc=True)
df_2023['timestamp'] = df_2023['timestamp'].dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)

# Spara den korrigerade filen
df_2023.to_csv('solis_data_2023_korrigerad.csv', index=False)
print("Filen för 2023 är nu korrigerad och sparad som 'solis_data_2023_korrigerad.csv'")

Filen för 2023 är nu korrigerad och sparad som 'solis_data_2023_korrigerad.csv'


In [15]:
import hmac, hashlib, base64, json, requests, time, os
import pandas as pd
from datetime import datetime, timezone, timedelta

# --- 1. KONFIGURATION (RÖR EJ SIGNERINGSLOGIKEN) ---
API_ID = "1300386381676697014" 
API_SECRET = "33aeed9e61334699a98ac08f6f6d851e"
INVERTER_SN = "1805040227120162"
BASE_URL = "https://www.soliscloud.com:13333"

def get_solis_headers(path, body_dict):
    body_str = json.dumps(body_dict, separators=(',', ':'))
    md5_hash = hashlib.md5(body_str.encode('utf-8')).digest()
    content_md5 = base64.b64encode(md5_hash).decode('utf-8')
    
    now = datetime.now(timezone.utc)
    date_str = now.strftime('%a, %d %b %Y %H:%M:%S GMT')
    
    content_type = "application/json;charset=UTF-8"
    sign_content_type = "application/json"
    
    # Exakt den radbrytningslogik som fungerade
    sign_parts = ["POST", content_md5, sign_content_type, date_str, path]
    sign_str = "\n".join(sign_parts)
    
    hashed = hmac.new(API_SECRET.encode('utf-8'), sign_str.encode('utf-8'), hashlib.sha1).digest()
    sign = base64.b64encode(hashed).decode('utf-8')
    
    return {
        "Content-MD5": content_md5,
        "Content-Type": content_type,
        "Date": date_str,
        "Authorization": f"API {API_ID}:{sign}"
    }, body_str

def fetch_solis_data(target_year):
    start_date = datetime(target_year, 1, 1)
    # Om det är innevarande år (2026), stoppa vid igår. Annars kör hela året.
    end_date = min(datetime(target_year, 12, 31), datetime.now() - timedelta(days=1))
    
    current_date = start_date
    all_days = []
    
    print(f"\n--- Startar hämtning för {target_year} ---")
    
    while current_date <= end_date:
        date_str = current_date.strftime('%Y-%m-%d')
        # Bestäm TZ-offset för anropet (2 för sommar, 1 för vinter i Sverige)
        # Detta hjälper API:et att skicka rätt dygnsfönster
        tz_offset = 2 if (datetime(target_year, 3, 25) < current_date < datetime(target_year, 10, 25)) else 1
        
        path = "/v1/api/inverterDay"
        payload = {"sn": INVERTER_SN, "money": "EUR", "time": date_str, "timeZone": tz_offset}
        
        headers, body = get_solis_headers(path, payload)
        
        try:
            res = requests.post(BASE_URL + path, headers=headers, data=body, timeout=15)
            if res.status_code == 200:
                data = res.json()
                if str(data.get("code")) == "0" and data.get("data"):
                    df_day = pd.DataFrame(data["data"])
                    all_days.append(df_day)
                    print(f"Hämtat: {date_str} ({len(data['data'])} punkter)", end="\r")
            else:
                print(f"\nFel vid {date_str}: HTTP {res.status_code}")
        except Exception as e:
            print(f"\nKritiskt fel vid {date_str}: {e}")
            
        current_date += timedelta(days=1)
        time.sleep(0.3) # Paus för att undvika rate-limit
    
    if all_days:
        full_df = pd.concat(all_days, ignore_index=True)
        
        # --- TIDSKORRIGERING TILL SVENSK TID ---
        # dataTimestamp är i millisekunder UTC
        full_df['timestamp'] = pd.to_datetime(full_df['dataTimestamp'].astype(float), unit='ms', utc=True)
        # Konvertera till Europe/Stockholm (hanterar sommar/vintertid perfekt)
        full_df['timestamp'] = full_df['timestamp'].dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)
        
        # Bearbeta värden
        full_df['pac'] = pd.to_numeric(full_df['pac'], errors='coerce')
        full_df.set_index('timestamp', inplace=True)
        
        # Aggregera till timvis data
        hourly_df = full_df['pac'].resample('1H').mean().reset_index()
        hourly_df.columns = ['timestamp', 'avg_power_kw']
        
        filename = f"solis_data_{target_year}_local.csv"
        hourly_df.to_csv(filename, index=False)
        print(f"\nKlar! Data för {target_year} sparad till {filename}")
        return hourly_df
    else:
        print(f"\nIngen data hittades för {target_year}.")
        return None

# --- KÖRNING FÖR 2024 OCH 2025 ---
data_2024 = fetch_solis_data(2024)
data_2025 = fetch_solis_data(2025)


--- Startar hämtning för 2024 ---


C:\Users\micha\AppData\Local\Temp\ipykernel_52932\4051666042.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_df = full_df['pac'].resample('1H').mean().reset_index()



Klar! Data för 2024 sparad till solis_data_2024_local.csv

--- Startar hämtning för 2025 ---
Hämtat: 2025-12-31 (100 punkter)
Klar! Data för 2025 sparad till solis_data_2025_local.csv


C:\Users\micha\AppData\Local\Temp\ipykernel_52932\4051666042.py:87: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_df = full_df['pac'].resample('1H').mean().reset_index()
